In [95]:
from clemcore.clemgame.benchmark import GameBenchmark
from clemcore.clemgame.master import GameMaster
from clemcore.backends.model_registry import Model
from typing import Dict, Tuple, List
import logging
from clemcore.clemgame.master import DialogueGameMaster, Player

In [96]:
logger = logging.getLogger(__name__)

In [97]:
class SnakeGameBenchmark(GameBenchmark):
    def __init__(self):
        super().__init__('SNAKE')

    def get_description(self):
        return "Plays a simple game of snake with one player."

    def is_single_player(self) -> bool:
        return True

    def create_game_master(self, experiment: Dict, player_models: List[Model]) -> GameMaster:
        return Snake(self.game_name, self.game_path, experiment, player_models)
        

In [98]:
import os
import sys
from dotenv import load_dotenv
load_dotenv(dotenv_path='./secrets.env')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

sys.path.append('/home/prigby/School/NLP/FinalProject/clemcore')


In [104]:
import random
from string import ascii_lowercase as letters
from typing import List

from clemcore.clemgame import Player

class Navigator(Player):
    def __init__(self, model: Model):
        super().__init__(model)

    def _custom_response(self, context):
        return f"<MOVE UP>"


class Describer(Player):
    def __init__(self, model: Model):
        super().__init__(model)

    def _custom_response(self, context):
        return "[][][*]\n[][][]\n[][s][]"
    

In [102]:
class Snake(DialogueGameMaster):
    """ Implements a game of snake in which one player describes the directions in which
    to move a snake within a grid to find its food.
    """
    
    def __init__(self, game_name: str, game_path: str, experiment: Dict, player_models: List[Model] = None):
        super().__init__(name, path, experiment, player_models)
        self.current_turn = 0
        self.gameboard = [[None] for _ in range(9)]
        self.invalid_response = False
        # READ IN SETTINGS FROM EXPERIMENT/INSTANCE.JSON???

    def _on_setup(self, **game_instance):
        # describer should be the programmatic player?
        self.navigator = Navigator(self.player_models[0])
        self.add_player(self.navigator)
        self.describer = Describer(self.player_models[1])
        self.add_player(self.describer)

        self.game_instance = game_instance
        self.max_turns = game_instance['max_turns']
        self.snake_location = game_instance['snake_start_loc']
        self.prey_location = game_instance['prey_start_loc']
        self.describer_tag = game_instance['describer_tag']
        self.navigator_tag = game_instance['navigator_tag']
        self.describer_initial_prompt = game_instance['describer_initial_prompt']
        self.navigator_initial_prompt = game_instance['navigator_initial_prompt']

    def _on_before_game(self):
        self.set_context_for(self.navigator, self.navigator_initial_prompt)

    def _does_game_proceed(self):
        """Proceed as long as the snake does not occupy the same gridspace as the prey."""
        if self.invalid_response:
            self.log_to_self("invalid response", "abort game")
            return False
        if self.win_state:
            self.log_to_self("game win", "end game")
            return False
        if self.invalid_state:
            self.log_to_self("game over", "end game")
            return False
        if self.current_turn >= self.max_turns:
            self.log_to_self("max turns reached", str(self.max_turns))
            return False
        return True

    def _validate_player_response(self, player: Player, utterance: str) -> bool:

        self.invalid_response = False
        self.win_state = False
        self.invalid_state = False
        
        if player == self.navigator:
            # is navigator response in valid format?    
            if not (utterance.startswith('<'.append(self.navigator_tag)) and utterance.endswith('>')):
                self.log_to_self("invalid format", "Invalid response.")
                self.invalid_format = True
                return False 
        # if player == self.describer:
            # if 
            #validate response format
            # pass
        else:
            return True

    def _on_valid_player_response(self, player: Player, parsed_response: str):
        if player == self.navigator:
            self.set_context_for(self.describer, parsed_response)
        elif player == self.describer:
            self.set_context_for(self.navigator, parsed_response)
            

In [103]:
from google import genai
from dotenv import load_dotenv
import os

In [83]:
history = []
history.append('USER: We are going to play a game of snake. The snake has no tail, and the snake cannot go outside the defined grid space. I will provide a grid with a starting position for the snake, and a position representing the "food". Your job is to tell me at each step what direction the snake will travel, in single-grid moves. The snake can move up, down, left, or right exactly one grid space per step. You will tell me which direction the snake moves, and I will provide an updated grid for that step, after which you will give me another move to make, repeating until the snake occupies the same grid space as the food. The snake will be represented using the character "s", and the food with character "*". You are ONLY allowed to responed with <MOVE: [UP | DOWN | LEFT | RIGHT]>.')

In [84]:
load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [85]:
response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=history[0]
)
history.append(f'MODEL: {response.text}')

In [86]:
for message in history:
    print(message)

USER: We are going to play a game of snake. The snake has no tail, and the snake cannot go outside the defined grid space. I will provide a grid with a starting position for the snake, and a position representing the "food". Your job is to tell me at each step what direction the snake will travel, in single-grid moves. The snake can move up, down, left, or right exactly one grid space per step. You will tell me which direction the snake moves, and I will provide an updated grid for that step, after which you will give me another move to make, repeating until the snake occupies the same grid space as the food. The snake will be represented using the character "s", and the food with character "*". You are ONLY allowed to responed with <MOVE: [UP | DOWN | LEFT | RIGHT]>.
MODEL: Understood. I will only respond with `<MOVE: [UP | DOWN | LEFT | RIGHT]>`.



In [87]:
history.append('USER: \n\t[][][*]\n\t[][][]\n\t[][][s]')

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=''.join(history)
)
history.append(f'MODEL: {response.text}')

In [88]:
for message in history:
    print(message)

USER: We are going to play a game of snake. The snake has no tail, and the snake cannot go outside the defined grid space. I will provide a grid with a starting position for the snake, and a position representing the "food". Your job is to tell me at each step what direction the snake will travel, in single-grid moves. The snake can move up, down, left, or right exactly one grid space per step. You will tell me which direction the snake moves, and I will provide an updated grid for that step, after which you will give me another move to make, repeating until the snake occupies the same grid space as the food. The snake will be represented using the character "s", and the food with character "*". You are ONLY allowed to responed with <MOVE: [UP | DOWN | LEFT | RIGHT]>.
MODEL: Understood. I will only respond with `<MOVE: [UP | DOWN | LEFT | RIGHT]>`.

USER: 
	[][][*]
	[][][]
	[][][s]
MODEL: <MOVE: UP>



In [89]:
history.append('USER:\n\t[][][*]\n\t[][][s]\n\t[][][]')

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=''.join(history)
)
history.append(f'MODEL: {response.text}')

In [90]:
for message in history:
    print(message)

USER: We are going to play a game of snake. The snake has no tail, and the snake cannot go outside the defined grid space. I will provide a grid with a starting position for the snake, and a position representing the "food". Your job is to tell me at each step what direction the snake will travel, in single-grid moves. The snake can move up, down, left, or right exactly one grid space per step. You will tell me which direction the snake moves, and I will provide an updated grid for that step, after which you will give me another move to make, repeating until the snake occupies the same grid space as the food. The snake will be represented using the character "s", and the food with character "*". You are ONLY allowed to responed with <MOVE: [UP | DOWN | LEFT | RIGHT]>.
MODEL: Understood. I will only respond with `<MOVE: [UP | DOWN | LEFT | RIGHT]>`.

USER: 
	[][][*]
	[][][]
	[][][s]
MODEL: <MOVE: UP>

USER:
	[][][*]
	[][][s]
	[][][]
MODEL: <MOVE: UP>



In [91]:
history.append('USER: \n\t[][][s]\n\t[][][]\n\t[][][]')
history.append('USER: Congrats! You win.')

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=''.join(history)
)
history.append(f'MODEL: {response.text}')

In [92]:
for message in history:
    print(message)

USER: We are going to play a game of snake. The snake has no tail, and the snake cannot go outside the defined grid space. I will provide a grid with a starting position for the snake, and a position representing the "food". Your job is to tell me at each step what direction the snake will travel, in single-grid moves. The snake can move up, down, left, or right exactly one grid space per step. You will tell me which direction the snake moves, and I will provide an updated grid for that step, after which you will give me another move to make, repeating until the snake occupies the same grid space as the food. The snake will be represented using the character "s", and the food with character "*". You are ONLY allowed to responed with <MOVE: [UP | DOWN | LEFT | RIGHT]>.
MODEL: Understood. I will only respond with `<MOVE: [UP | DOWN | LEFT | RIGHT]>`.

USER: 
	[][][*]
	[][][]
	[][][s]
MODEL: <MOVE: UP>

USER:
	[][][*]
	[][][s]
	[][][]
MODEL: <MOVE: UP>

USER: 
	[][][s]
	[][][]
	[][][]
USE